In [4]:
!apt-get install -y -q zstd
print("zstd algo installed successfully")

Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
zstd algo installed successfully


In [5]:
!curl -fsSL https://ollama.com/install.sh | sh
print("Ollama installed successfully")

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama installed successfully


In [6]:
!pip install -q ollama faiss-cpu sentence-transformers datasets langchain langchain-community langchain-ollama
print("✅ Python libraries installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
✅ Python libraries installed


In [7]:
import subprocess
import time
import requests

# Start Ollama server
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("Waiting for Ollama server...")

# Wait for server to be ready
for i in range(30):
    try:
        response = requests.get("http://localhost:11434")
        if response.status_code == 200:
            print("✅ Ollama server is running!")
            break
    except Exception:
        time.sleep(2)
else:
    print("❌ Server did not start - re-run this cell once")

Waiting for Ollama server...
✅ Ollama server is running!


In [8]:
print("Pulling mistral (~4GB)")
!ollama pull mistral
print("mistral installed successfully")

Pulling mistral (~4GB)

mistral installed successfully


In [9]:
import ollama
response_example =ollama.chat(
    model= "mistral",
    messages= [
        {
            "role":"user",
            "content":"In one sentence, what is HR policy"
        }
        ]
)

print(response_example["message"]["content"])

 An HR policy is a set of guidelines or principles established by an organization to govern the behavior and management of its employees in areas such as recruitment, compensation, benefits, and workplace conduct.


In [10]:
conversation_history = []
print("Chat with Mistral (type : Quit to exit)")

while True:
  user_input = input("You: ").strip()

  if user_input.lower() in ["quit","exit","q","ronak"]:
    print("Chat ended.")
    break
  if not user_input:
    continue

  conversation_history.append({
      "role":"user",
      "content": user_input
  })

  response_demo = ollama.chat(model="mistral", messages= conversation_history)

  replay = response_demo["message"]["content"]
  conversation_history.append({
      "role":"assistant",
      "content":replay
  })

  print(f"\n Mistral: {replay} \n")

Chat with Mistral (type : Quit to exit)
You: q
Chat ended.


In [11]:
import pandas as pd
from datasets import load_dataset

print("Loading ECHR legal case law from HuggingFace...")
DATASET_LOADED = False

try:
    ds = load_dataset("coastalcph/lex_glue", "ecthr_a", split="train", trust_remote_code=True)
    df = ds.to_pandas()[["text"]].dropna()
    df["text"] = df["text"].apply(lambda x: " ".join(x) if isinstance(x, list) else str(x))
    df = df[df["text"].str.len() > 300].head(500).reset_index(drop=True)
    df["doc_id"] = [f"ECHR_Case_{i}" for i in range(len(df))]
    print(f"✅ Loaded {len(df)} real ECHR court cases")
    DATASET_LOADED = True
except Exception as e:
    print(f"⚠️  HuggingFace dataset unavailable ({e}) — using built-in corpus")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'coastalcph/lex_glue' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'coastalcph/lex_glue' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading ECHR legal case law from HuggingFace...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

ecthr_a/train-00000-of-00001.parquet:   0%|          | 0.00/42.4M [00:00<?, ?B/s]

...

ecthr_a/validation-00000-of-00001.parque(…):   0%|          | 0.00/5.26M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

✅ Loaded 500 real ECHR court cases


In [12]:
if not DATASET_LOADED:
    legal_docs = data = [
    {
        "doc_id": "Miranda_v_Arizona_1966",
        "text": "Miranda v. Arizona, 384 U.S. 436 (1966). The Supreme Court held that statements made during custodial interrogation are admissible only if the suspect is informed of rights: remain silent, statements can be used in court, right to an attorney, and appointed attorney if indigent."
    },
    {
        "doc_id": "Brown_v_Board_of_Education_1954",
        "text": "Brown v. Board of Education, 347 U.S. 483 (1954). The Court declared racial segregation in public schools unconstitutional, overturning Plessy v. Ferguson and holding that separate facilities are inherently unequal."
    },
    {
        "doc_id": "Marbury_v_Madison_1803",
        "text": "Marbury v. Madison (1803) established judicial review, giving courts power to strike down unconstitutional laws."
    },
    {
        "doc_id": "Fourth_Amendment_Search_Seizure",
        "text": "The Fourth Amendment protects against unreasonable searches and seizures. Warrants require probable cause. Exceptions include consent, exigent circumstances, search incident to arrest, automobile exception, and stop-and-frisk."
    },
    {
        "doc_id": "Negligence_Tort_Law",
        "text": "Negligence requires duty, breach, causation, and damages. Duty applies to foreseeable plaintiffs. The reasonable person standard applies. The eggshell skull rule makes defendants liable for full harm."
    },
    {
        "doc_id": "Contract_Law_Formation",
        "text": "Contracts require offer, acceptance, and consideration. Acceptance must match the offer. Consideration must have value. Promissory estoppel may apply where reliance exists."
    },
    {
        "doc_id": "Employment_Discrimination_Title_VII",
        "text": "Title VII prohibits discrimination based on race, color, religion, sex, or national origin. Covers disparate treatment and disparate impact. Includes sexual harassment protections."
    },
    {
        "doc_id": "Corporate_Fiduciary_Duties",
        "text": "Directors owe duty of care and loyalty. Business judgment rule protects decisions unless there is fraud or conflict of interest."
    },
    {
        "doc_id": "Bankruptcy_Chapter7_Chapter13",
        "text": "Chapter 7 allows liquidation of assets to discharge debt. Chapter 13 allows repayment plans. Some debts like taxes and child support are non-dischargeable."
    },
    {
        "doc_id": "Copyright_Law",
        "text": "Copyright protects original works. Rights include reproduction and distribution. Fair use depends on purpose, nature, amount, and market impact."
    },
    {
        "doc_id": "ECHR_Article6_Fair_Trial",
        "text": "Article 6 guarantees the right to a fair and public hearing by an independent tribunal, including legal representation and examination of witnesses."
    },
    {
        "doc_id": "Asylum_Law_US",
        "text": "Asylum requires proof of persecution based on race, religion, nationality, social group, or political opinion. Must file within one year unless exceptions apply."
    },
    {
        "doc_id": "Dobbs_v_Jackson_2022",
        "text": "Dobbs v. Jackson (2022) overturned Roe v. Wade, holding the Constitution does not guarantee a right to abortion and returning authority to states."
    },
    {
        "doc_id": "Gideon_v_Wainwright_1963",
        "text": "Gideon v. Wainwright (1963) established the right to counsel for criminal defendants who cannot afford an attorney."
    }
]
    df = pd.DataFrame(legal_docs)
    print(f"✅ Built-in legal corpus ready: {len(df)} documents")

print(df["doc_id"].tolist())


['ECHR_Case_0', 'ECHR_Case_1', 'ECHR_Case_2', 'ECHR_Case_3', 'ECHR_Case_4', 'ECHR_Case_5', 'ECHR_Case_6', 'ECHR_Case_7', 'ECHR_Case_8', 'ECHR_Case_9', 'ECHR_Case_10', 'ECHR_Case_11', 'ECHR_Case_12', 'ECHR_Case_13', 'ECHR_Case_14', 'ECHR_Case_15', 'ECHR_Case_16', 'ECHR_Case_17', 'ECHR_Case_18', 'ECHR_Case_19', 'ECHR_Case_20', 'ECHR_Case_21', 'ECHR_Case_22', 'ECHR_Case_23', 'ECHR_Case_24', 'ECHR_Case_25', 'ECHR_Case_26', 'ECHR_Case_27', 'ECHR_Case_28', 'ECHR_Case_29', 'ECHR_Case_30', 'ECHR_Case_31', 'ECHR_Case_32', 'ECHR_Case_33', 'ECHR_Case_34', 'ECHR_Case_35', 'ECHR_Case_36', 'ECHR_Case_37', 'ECHR_Case_38', 'ECHR_Case_39', 'ECHR_Case_40', 'ECHR_Case_41', 'ECHR_Case_42', 'ECHR_Case_43', 'ECHR_Case_44', 'ECHR_Case_45', 'ECHR_Case_46', 'ECHR_Case_47', 'ECHR_Case_48', 'ECHR_Case_49', 'ECHR_Case_50', 'ECHR_Case_51', 'ECHR_Case_52', 'ECHR_Case_53', 'ECHR_Case_54', 'ECHR_Case_55', 'ECHR_Case_56', 'ECHR_Case_57', 'ECHR_Case_58', 'ECHR_Case_59', 'ECHR_Case_60', 'ECHR_Case_61', 'ECHR_Case_62', '

In [13]:
!pip install -q -U langchain langchain-community langchain-text-splitters langchain-ollama langchain-huggingface
print("langchain installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 13.3 MB/s eta 0:00:00
langchain installed successfully


In [14]:
#for debugging purpose

import langchain
print(langchain.__version__)

# This will show you where the class actually lives now
import subprocess
result = subprocess.run(["python", "-c", "import langchain_text_splitters; print('ok')"], capture_output=True, text=True)
print(result.stdout, result.stderr)


1.2.17
ok
 


In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size = 450, chunk_overlap = 80)

documents = []

for _,row in df.iterrows():
  for i,chunk in enumerate(splitter.split_text(row["text"])):
    documents.append(Document(
        page_content=chunk,
        metadata = {"source":row["doc_id"],"chunk":i}
    ))
print(f"{len(documents)} chunks from {len(df)} documents")

print("Embedding on GPU...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = FAISS.from_documents(documents, embeddings)
vectorstore.save_local("/content/legal_faiss")
print(f"FAISS index ready - {vectorstore.index.ntotal} vectors")


14411 chunks from 500 documents
Embedding on GPU...


/tmp/ipykernel_5235/2505131622.py:20: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index ready - 14411 vectors


In [16]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="mistral", temperature=0.1, num_predict=400)

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

prompt = PromptTemplate.from_template("""
You are an expert legal research assistant.
Answer the question using ONLY the legal documents provided below.
Cite the document names in your answer when referencing specific cases or principles.
If the documents do not contain enough information, say so clearly.

Legal Documents:
{context}

Question: {question}

Legal Answer:""")

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

print("✅ RAG chain ready — Mistral + FAISS + Local Embeddings")

✅ RAG chain ready — Mistral + FAISS + Local Embeddings


In [17]:
def ask(question: str):
    print(f"\n{'='*65}")
    print(f"❓ {question}")
    print(f"{'='*65}")
    docs = retriever.invoke(question)           # ← was get_relevant_documents
    sources = list(dict.fromkeys(d.metadata["source"] for d in docs))
    print(f"📚 Sources pulled: {', '.join(sources)}")
    print("-"*65)
    answer = rag_chain.invoke(question)
    print(f"⚖️  {answer}")
    return answer

_ = ask("What rights must police inform a suspect of before interrogation?")



❓ What rights must police inform a suspect of before interrogation?
📚 Sources pulled: ECHR_Case_30, ECHR_Case_199
-----------------------------------------------------------------
⚖️   Based on the provided documents, specifically [ECHR_Case_30], it is not explicitly stated what specific rights the police must inform a suspect of before interrogation. However, it does emphasize that the arresting officer must have reasonable cause to suspect before exercising the power of arrest, and this reasonable suspicion may be based on information which has been given to him.

In [ECHR_Case_199], there is a mention of an interrogation where the police attempted to "rattle" or unsettle the suspect, but it does not provide details about the rights that should have been informed to the suspect before this interrogation.

Therefore, without more specific information from other legal documents, it cannot be definitively answered what rights police must inform a suspect of before interrogation.


In [21]:
!pip install -q gradio
print("Gradio install successfully")

Gradio install successfully


In [22]:
import gradio as gr


# ── helpers ──────────────────────────────────────────────────
def get_sources(docs):
    return list(dict.fromkeys(d.metadata["source"] for d in docs))


def rag_response(message, history):
    docs   = retriever.invoke(message)
    sources = get_sources(docs)
    context = "\n\n".join(
        f"[{d.metadata['source']}]\n{d.page_content}" for d in docs
    )


    prompt_text = f"""You are an expert legal research assistant.
Answer the question using ONLY the legal documents provided below.
Cite the document names in your answer when referencing specific cases or principles.
If the documents do not contain enough information, say so clearly.


Legal Documents:
{context}


Question: {message}


Legal Answer:"""


    response = ollama.chat(
        model="mistral",
        messages=[{"role": "user", "content": prompt_text}]
    )
    answer = response["message"]["content"]


    source_line = "\n\n---\n📚 **Sources:** " + " · ".join(f"`{s}`" for s in sources)
    return answer + source_line


# ── UI ───────────────────────────────────────────────────────
with gr.Blocks(title="Legal RAG", theme=gr.themes.Soft()) as demo:


    gr.Markdown("# Legal RAG — Powered by Mistral + FAISS\nAsk any legal question. Answers are grounded in the case law documents.")


    chatbot = gr.Chatbot(
        label="Legal Assistant",
        height=480,
        show_copy_button=True,
        render_markdown=True,
    )
    with gr.Row():
        txt = gr.Textbox(
            placeholder="e.g. What are Miranda rights?",
            show_label=False,
            scale=9,
        )
        btn = gr.Button("Ask", scale=1, variant="primary")


    gr.Examples(
        examples=[
            "What rights must police inform a suspect of before interrogation?",
            "What are the elements required to prove negligence?",
            "What is the difference between Chapter 7 and Chapter 13 bankruptcy?",
            "What fiduciary duties do corporate directors owe to shareholders?",
            "How does the Fourth Amendment protect against unreasonable searches?",
            "Can corporations spend unlimited money on political campaigns?",
        ],
        inputs=txt,
    )


    def respond(message, chat_history):
        answer = rag_response(message, chat_history)
        chat_history.append((message, answer))
        return "", chat_history


    txt.submit(respond, [txt, chatbot], [txt, chatbot])
    btn.click(respond,  [txt, chatbot], [txt, chatbot])


demo.launch(share=True, quiet=True)

/tmp/ipykernel_5235/1770370748.py:45: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Legal RAG", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_5235/1770370748.py:51: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_5235/1770370748.py:51: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_5235/1770370748.py:51: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to e

* Running on public URL: https://02c020294bb18a4dd8.gradio.live
